In [ ]:
import torch
import matplotlib.pyplot as plt
from spatialca import map, game, net, metric
import time
import matplotlib.animation as animation
import pandas as pd
from matplotlib.lines import Line2D
import collections
import functools
import copy
import numpy as np
import os
import seaborn as sns
from scipy import stats

## Data

In [ ]:
repetitions = 3

COLOUR_MAP = map.satellite_colour_map
num_classes = len(COLOUR_MAP)
_scale_down = 3
_n_splits = 3
num_agents = 20

_place = "rio"
cities = [_place]
data_source = f"openearth_map_train_data_{_place}"
og_img_dir = f"./openearth/{_place}/images"
img_dir = f"./openearth/{_place}/labels"
data_file_path = f"./data/openearth_map_train_data_{_place}.json"

real_images = [f"{og_img_dir}/{f}" for f in os.listdir(og_img_dir)]
seg_images = [f"{img_dir}/{f}" for f in os.listdir(img_dir)]

real_image_paths = real_images[0:2]
train_image_paths = seg_images[0:2]
test_image_paths = seg_images[2:3]

In [ ]:
real_images = [map.load_rgb_image(image_path) for image_path in real_image_paths]
fig, axes = plt.subplots(1, len(real_images), figsize=(len(real_images) * 4, 4))

for i, rgb_image in enumerate(real_images):
    axes[i].imshow(rgb_image, cmap="gray")
    axes[i].set_title(f"Real {_place.capitalize()} {i + 1}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()

seg_images = [
    map.load_openearth_seg_image(image_path, color_map=COLOUR_MAP)
    for image_path in train_image_paths
]
fig, axes = plt.subplots(1, len(seg_images), figsize=(len(seg_images) * 4, 4))

for i, rgb_image in enumerate(seg_images):
    axes[i].imshow(rgb_image, cmap="gray")
    axes[i].set_title(f"Seg. {_place.capitalize()} {i + 1}")

plt.tight_layout()
plt.show()

train_images = [
    map.load_openearth_seg_image(
        image_path, scale_down=_scale_down, color_map=COLOUR_MAP
    )
    for image_path in train_image_paths
]
train_images = [
    img for image in train_images for img in map.split_image(image, _n_splits)
]
rows = (len(train_images) + 7) // 9  # Calculate the number of rows needed
fig, axes = plt.subplots(rows, 9, figsize=(32, rows * 4))

for idx, rgb_image in enumerate(train_images):
    row, col = divmod(idx, 9)
    axes[row, col].imshow(rgb_image, cmap="gray")
    axes[row, col].set_title(f"Train Seg. {_place.capitalize()} {idx + 1}")

plt.tight_layout()
plt.show()

In [ ]:
images_to_plot = [
    real_images[0],
    seg_images[0],
    train_images[0],
]

# side by side
fig, axes = plt.subplots(1, len(images_to_plot), figsize=(len(images_to_plot) * 4, 4))

for i, (rgb_image, name) in enumerate(
    zip(images_to_plot, ["Original Image", "Segmentation Image", "Training Image"])
):
    axes[i].imshow(rgb_image)
    axes[i].set_title(name)

plt.tight_layout()
plt.show()

In [ ]:
COLOUR_MAP = map.synthetic_colours
num_classes = 4
num_agents = 10
repetitions = 1
maps_per_city = 3
cities = ["farmland", "river", "archipelago"] * maps_per_city
data_source = "map_train_data_synth_all"

In [ ]:
place_to_eval = cities

n_cities = len(cities)
real_images = [
    map.generate_synthetic_satellite_image(city, COLOUR_MAP) for city in cities
]
fig, axes = plt.subplots(1, n_cities, figsize=(n_cities * 3, 3))
for i, rgb_image in enumerate(real_images):
    axes[i].imshow(rgb_image, cmap="gray")
    axes[i].set_title(f"Map {i + 1}")
plt.tight_layout()
plt.show()

train_images = [
    map.generate_synthetic_satellite_image(city, COLOUR_MAP) for city in place_to_eval
]

In [ ]:
n_cities = len(cities)
if _place in cities:
    train_images = [
        map.load_openearth_seg_image(
            image_path, scale_down=_scale_down, color_map=COLOUR_MAP
        )
        for image_path in test_image_paths
    ]
    train_images = [
        img for image in train_images for img in map.split_image(image, _n_splits)
    ]
    _names = [f"{_place}-{i + 1}" for i in range(len(train_images))]
else:
    _config = map.SyntheticMapConfig(size=64)  # 64
    train_images = [
        map.generate_synthetic_satellite_image(city, COLOUR_MAP, _config)
        for city in cities
    ]
    _names = [f"{city}-1" for city in cities]
train_images = train_images[0:3]

if len(train_images) == 1:
    plt.figure(figsize=(3, 3))
    plt.imshow(train_images[0], cmap="gray")
    plt.title("Map 1")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
else:
    fig, axes = plt.subplots(1, len(train_images), figsize=(len(train_images) * 3, 3))
    for i, (rgb_image, city) in enumerate(zip(train_images, _names)):
        axes[i].imshow(rgb_image, cmap="gray")
        axes[i].set_title(f"Map {i + 1} - {city}")
        axes[i].axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
extrapolator_movement_models = {
    "NeuralCA.t=0": net.NeuralCA.load(f"best_chkpt_CNN-{data_source}.pth"),
    "NeuralCA.t=18": net.NeuralCA.load(f"best_chkpt_NeuralCA-{data_source}.pth"),
    "UNet": net.UNet.load(f"best_chkpt_UNet-{data_source}.pth"),
}
extrapolator_models = extrapolator_movement_models
# extrapolator_movement_models = {}

simulation_steps = 150
cities = _names * repetitions
train_images = [img for img in train_images for _ in range(repetitions)]
agent_view_radius = 2
movements = ["frontier", "greedy", "random"]

all_images = {f"{_place}-train-{i + 1}": img for i, img in enumerate(train_images)}
images_to_test = all_images
print(f"Testing on {len(images_to_test)} images")
print(all_images.keys())

In [ ]:
game_envs: dict[dict[str, game.MultiAgentMapper]] = {}
for name, img in images_to_test.items():
    game_envs[name] = []
    current_env_results = {}
    agent_inits = game.init_agents(img.shape[0], img.shape[1], num_agents)

    for model in extrapolator_movement_models:
        _time = time.strftime("%H:%M:%S")
        print(f"{_time} - Starting {name} - extrapolator {model}")
        guided_model = {model: extrapolator_movement_models[model]}
        _env = game.MultiAgentMapper(
            torch.from_numpy(img),
            num_agents=num_agents,
            agent_view_radius=agent_view_radius,
            movement_style="extrapolator",
            colour_map=COLOUR_MAP,
            agent_attrs=copy.deepcopy(agent_inits),
        )
        _env.simulate(guided_model, steps=simulation_steps)
        current_env_results[f"extrapolator-{model}"] = _env

    for movement in movements:
        _time = time.strftime("%H:%M:%S")
        print(f"{_time} - Starting {name} - {movement}")
        _env = game.MultiAgentMapper(
            torch.from_numpy(img),
            num_agents=num_agents,
            agent_view_radius=agent_view_radius,
            movement_style=movement,
            colour_map=COLOUR_MAP,
            agent_attrs=copy.deepcopy(agent_inits),
        )
        _env.simulate(extrapolator_models, steps=simulation_steps)
        current_env_results[movement] = _env

    game_envs[name].append(current_env_results)

# Analyse

In [ ]:
movement_colours = ["b", "g", "r", "purple", "yellow"]
linestyles = ["-", "--", ":", "-."]


def make_legend(ax):
    linelegend = functools.partial(Line2D, [0], [0], lw=2)
    color_legend = [
        linelegend(color=movement_colours[0], label="Greedy"),
        linelegend(color=movement_colours[1], label="Frontier"),
        linelegend(color=movement_colours[2], label="Random"),
        linelegend(color=movement_colours[3], label="Extrapolator"),
        # linelegend(color=movement_colours[4], label="Error"),
    ]
    color_legend_ = ax.legend(
        handles=color_legend, title="Movements", loc="center right"
    )
    ax.add_artist(color_legend_)
    style_legend = [
        linelegend(linestyle=linestyles[1], color="black", label="UNet"),
        linelegend(linestyle=linestyles[2], color="black", label="NeuralCA.t=18"),
        linelegend(linestyle=linestyles[3], color="black", label="NeuralCA.t=0"),
    ]
    ax.legend(handles=style_legend, title="Models", loc="lower right")
    return ax


threshold_val = 1 / num_classes  # 0.9
masked_metric_func = functools.partial(
    metric.mask_coverage, threshold_val=threshold_val
)

envs_to_do = list(game_envs.keys())
all_movements = [
    *movements,
    *[f"extrapolator-{model}" for model in extrapolator_movement_models.keys()],
    # *[f"error-{model}" for model in extrapolator_movement_models.keys()],
]

In [ ]:
all_move_results = {}

for _env_name in envs_to_do:
    data = {}

    for move in all_movements:
        models_to_load = list(extrapolator_models.keys())
        if "extrapolator" in move:
            models_to_load = [move.split("extrapolator-")[1]]

        game_env_results = [g[move] for g in game_envs[_env_name]]

        unique_movements = []
        all_inter_agent_distances = []

        for repeat_env in game_env_results:
            flattened_moves = [m for g_move in repeat_env.history_moves for m in g_move]
            move_counts = collections.Counter(flattened_moves)
            frequent_moves = [move for move, count in move_counts.items() if count >= 2]
            unique_movements.append(len(frequent_moves))

        data[move] = float(np.mean(unique_movements, axis=0))

    all_move_results[_env_name] = data

In [ ]:
all_coverage_results = {}
for _env_name in envs_to_do:
    data = {}
    for move in all_movements:
        models_to_load = list(extrapolator_models.keys())
        for x in ["extrapolator", "error"]:
            if x in move:
                models_to_load = [move.split(f"{x}-")[1]]

        game_env_results = [g[move] for g in game_envs[_env_name]]

        observation_metrics = []
        for repeat_env in game_env_results:
            oh_target = map.rgb_to_target_tensor(repeat_env.raw_map, COLOUR_MAP)
            repeat_results = []
            for frame in repeat_env.history_maps:
                _value = (frame[:, :, -1] < 0.5).sum() / frame[:, :, -1].numel()
                repeat_results.append(_value)
            observation_metrics.append(repeat_results)
        data[f"Observations - {move}"] = np.mean(observation_metrics, axis=0)

        for model_name in models_to_load:
            refined_metrics = []
            for repeat_env in game_env_results:
                repeat_results = []
                oh_target = map.rgb_to_target_tensor(repeat_env.raw_map, COLOUR_MAP)
                for frame, pred in zip(
                    repeat_env.history_maps, repeat_env.history_refineds[model_name]
                ):
                    _value = masked_metric_func(
                        pred[:num_classes], oh_target, frame[:, :, -1]
                    )
                    repeat_results.append(_value)
                refined_metrics.append(repeat_results)
            data[f"{model_name} - {move}"] = np.mean(refined_metrics, axis=0)
    all_coverage_results[_env_name] = data

In [ ]:
time_to_coverage_values = [0.5, 0.75, 0.90, 0.99]
env_data = {}
for _env_name, data in all_coverage_results.items():
    rows = []
    for k, series in data.items():
        model, movement = k.split(" - ")
        data_row = {"Model": model.strip(), "Movement": movement.strip()}
        for t in time_to_coverage_values:
            time_at_t = "N/A"
            above = [i + 1 for i, x in enumerate(series) if t <= x]
            if above:
                time_at_t = above[0]
            data_row[f"Time to {int(100 * t)}%"] = time_at_t
        rows.append(data_row)
    env_data[_env_name] = rows

In [ ]:
all_rows = []
for _env_name in env_data:
    curr_rows = env_data[_env_name]
    for matching_row in curr_rows:
        matching_row["Image"] = _env_name
        matching_row["Env"] = _env_name.split("-")[0]
    all_rows += env_data[_env_name]
df = pd.DataFrame(all_rows)

# Visualise

In [ ]:
move_df = pd.DataFrame(all_move_results)
means = move_df.mean(axis=1)
stds = move_df.std(axis=1)

plt.figure(figsize=(8, 4))

bars = plt.bar(move_df.index, means, yerr=stds, capsize=6, edgecolor="black")
for bar, mean in zip(bars, means):
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2.0,
        height * 0.3,  # position above error bar
        f"{mean:.0f}",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold",
    )

plt.ylabel("# Redundant Visits", fontsize=14)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.grid(axis="y", linestyle="--", linewidth=0.5)
plt.show()

In [ ]:
df["Movement"] = df["Movement"].apply(
    lambda x: "Extrapolator" if "extrapolator" in x else "Error" if "error" in x else x
)
df["Movement"] = df["Movement"].str.capitalize()

In [ ]:
images_to_analyse = [img_n for img_n in df["Image"].unique() if cities[0] in img_n]
images_to_analyse = [img_n for img_n in df["Image"].unique() if cities[1] in img_n]
images_to_analyse = [img_n for img_n in df["Image"].unique() if cities[2] in img_n]

images_to_analyse = [img_n for img_n in df["Image"].unique()]

print(images_to_analyse)
df_for_plotting = df[df["Image"].isin(images_to_analyse)]

In [ ]:
mean_data = []
for index, row in df_for_plotting.iterrows():
    base = {"Model": row["Model"], "Movement": row["Movement"].capitalize()}
    for t in time_to_coverage_values:
        col = f"Time to {int(t * 100)}%"
        entry = base.copy()
        entry.update(
            {
                "Threshold": col,
                "Time": row[col],
                "Env": row["Env"],
                "Image": row["Image"],
            }
        )
        mean_data.append(entry)

df_melted = pd.DataFrame(mean_data)
df_melted["Time"] = pd.to_numeric(df_melted["Time"], errors="coerce")

In [ ]:
sns.set_theme(style="whitegrid")
palette = sns.color_palette("Set2", n_colors=len(df_melted["Model"].unique()))
thresholds = df_melted["Threshold"].unique()
n_metric = len(thresholds)

fig, axes = plt.subplots(
    nrows=(n_metric + 1) // 2, ncols=2, figsize=(16, 8), constrained_layout=True
)
axes = axes.flatten()
for ax, threshold in zip(axes, thresholds):
    sub_df = df_melted[df_melted["Threshold"] == threshold]
    x_labels = sub_df["Movement"].unique()
    models = sub_df["Model"].unique()
    x = np.arange(len(x_labels))
    width = 0.8 / len(models)

    for j, model in enumerate(models):
        model_data = sub_df[sub_df["Model"] == model]
        times = []
        errors = []
        for label in x_labels:
            matching_row = model_data[model_data["Movement"] == label]
            valid_times = []
            _mean_time = np.nan
            _error = np.nan
            if not matching_row.empty:
                valid_times = [x for x in matching_row["Time"].values if x != "N/A"]
                if valid_times:
                    if label == "Extrapolator" and model == "Observations":
                        _mean_time = np.nan
                    else:
                        _mean_time = sum(valid_times) / len(valid_times)
                    if len(valid_times) > 1:
                        _error = np.std(valid_times) * 1.96 / np.sqrt(len(valid_times))
            times.append(_mean_time)
            errors.append(_error)

        positions = x - (0.8 / 2) + j * width
        ax.bar(
            positions,
            times,
            width,
            yerr=errors,
            label=model,
            color=palette[j],
            capsize=5,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(x_labels)
    ax.set_title(f"Coverage Threshold: {threshold}", fontsize=12)
    ax.set_xlabel("")
    ax.set_ylabel("Timesteps")
    ax.set_ylim(
        0,
    )

    if threshold == thresholds[-1]:
        ax.legend()

fig.suptitle("Time to Coverage $\pm$ 95$\%$ CI by Threshold", fontsize=16)
plt.show()

In [ ]:
# assuming df is your DataFrame


def mean_ci(series):
    series = pd.to_numeric(series, errors="coerce").dropna()
    n = len(series)
    if n == 0:
        return "N/A"
    m = series.mean()
    if n == 1:
        return f"{m:.2f}"
    ci = stats.sem(series) * 1.96
    return f"{m:.0f} ± {ci:.1f}"


# compute mean ± CI for each group
summary = (
    df.groupby(["Model", "Movement"])[
        ["Time to 50%", "Time to 75%", "Time to 90%", "Time to 99%"]
    ]
    .apply(lambda g: g.apply(mean_ci))
    .reset_index()
)
print(summary.to_markdown(index=False))

In [ ]:
movements_to_plot = []
things_to_plot = {}
for _env_name, data in all_coverage_results.items():
    if _env_name not in images_to_analyse:
        continue
    for k, series in data.items():
        if movements_to_plot and all(
            m.lower() not in k.lower() for m in movements_to_plot
        ):
            continue
        if "obs" in k.lower():
            continue
        if k not in things_to_plot:
            things_to_plot[k] = []
        things_to_plot[k].append(series)

In [ ]:
fig, ax = plt.subplots(figsize=(21, 7))
ax = make_legend(ax)

ax.set_xlabel("Timesteps")
ax.set_ylabel("Coverage $\pm$ 95% CI")
for k, series_list in things_to_plot.items():
    series = np.mean(series_list, axis=0)
    st_dev = np.std(series_list, axis=0) * 1.96 / np.sqrt(len(series_list))
    if "observations" in k.lower():
        lstyle = linestyles[0]
    elif "unet" in k.lower():
        lstyle = linestyles[1]
    elif "ca" in k.lower():
        lstyle = linestyles[2]
    elif "t=0" in k.lower():
        lstyle = linestyles[3]
    else:
        raise ValueError(f"Unknown model name in {k}")

    if "greedy" in k.lower():
        color = movement_colours[0]
    elif "frontier" in k.lower():
        color = movement_colours[1]
    elif "random" in k.lower():
        color = movement_colours[2]
    elif "extrapolator" in k.lower():
        color = movement_colours[3]
    elif "error" in k.lower():
        color = movement_colours[4]
    else:
        raise ValueError(f"Unknown movement name in {k}")
    plt.plot(series, color=color, linestyle=lstyle)
    plt.fill_between(
        range(len(series)),
        series - st_dev,
        series + st_dev,
        color=color,
        alpha=0.2,
    )
plt.show()

## Videos

In [ ]:
RECORD_ROLLOUTS = True
record_interval = 3
total_frames = int(simulation_steps / 2)
plot_errors = False
prefix = "./videos/mapping"
img_name = images_to_analyse[-3]

movement = "random"
movement = "greedy"
movement = "frontier"
# movement = "extrapolator-UNet"
movement = "extrapolator-NeuralCA"

model_name = "UNet"
model_name = "NeuralCA"

game_env = [g[movement] for g in game_envs[img_name]][0]
frame_source = game_env.history_maps

# frame_source = game_env.history_refineds[model_name]
# frame_source = game_env.history_sigmas[model_name]

In [ ]:
if RECORD_ROLLOUTS:
    path = f"{prefix}_rollout.gif"

    def update_frame(i):
        snapshot = frame_source[i]
        if snapshot.shape[0] == num_classes + 1:
            snapshot = map.softmax_to_rgb(snapshot[:num_classes], COLOUR_MAP)
        elif len(snapshot.shape) == 2:
            snapshot = snapshot.detach().numpy()
        else:
            snapshot = snapshot[:, :, :3].detach().numpy()
        agent_positions = game_env.history_moves[i]
        ax.clear()
        ax.imshow(snapshot)
        for pos in agent_positions:
            ax.plot(pos[1], pos[0], "D", c="white", markersize=5)
        ax.set_title(f"Step {i}")
        ax.axis("off")
        return (ax,)

    fig, ax = plt.subplots(figsize=(8, 8))
    ani = animation.FuncAnimation(
        fig,
        update_frame,
        frames=range(0, total_frames, record_interval),
        interval=400,
    )
    ani.save(path, writer="pillow")